In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Teknik mitigasi dan supresi error

> **Note:** Rilis beta dari model eksekusi baru kini tersedia. Model eksekusi terarah memberikan fleksibilitas lebih besar saat mengustomisasi alur kerja mitigasi error kamu. Lihat panduan [Model eksekusi terarah](/guides/directed-execution-model) untuk informasi lebih lanjut.


<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan untuk menggunakan versi ini atau yang lebih baru.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Teknik mitigasi error dan supresi error digunakan untuk meningkatkan kualitas hasil saat memperbesar skala ke beban kerja yang lebih besar. Halaman ini memberikan penjelasan tingkat tinggi tentang teknik supresi error dan mitigasi error yang tersedia melalui Qiskit Runtime.

Sel berikut mengimpor primitif Estimator dan membuat Backend yang akan digunakan untuk menginisialisasi Estimator di sel kode berikutnya.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Dynamical decoupling
Circuit kuantum dieksekusi pada perangkat keras IBM&reg; sebagai rangkaian pulsa gelombang mikro yang perlu dijadwalkan dan dijalankan pada interval waktu yang tepat.
Sayangnya, interaksi yang tidak diinginkan antar Qubit dapat menyebabkan error koheren pada Qubit yang sedang diam. Dynamical decoupling bekerja dengan menyisipkan rangkaian pulsa pada Qubit yang diam untuk kira-kira membatalkan efek error tersebut. Setiap rangkaian pulsa yang disisipkan setara dengan operasi identitas, namun keberadaan fisik pulsa tersebut berfungsi untuk menekan error.
Ada banyak pilihan rangkaian pulsa, dan rangkaian mana yang lebih baik untuk setiap kasus tertentu masih menjadi [area penelitian aktif](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Perlu dicatat bahwa dynamical decoupling terutama berguna untuk Circuit yang mengandung celah di mana beberapa Qubit duduk diam tanpa operasi apa pun yang bekerja pada mereka. Jika operasi dalam Circuit sangat padat, sehingga semua Qubit sibuk sebagian besar waktu, maka penambahan pulsa dynamical decoupling mungkin tidak meningkatkan performa. Bahkan, hal itu bisa memperburuk performa karena ketidaksempurnaan pada pulsa itu sendiri.

Diagram di bawah ini menggambarkan dynamical decoupling dengan rangkaian pulsa XX. Circuit abstrak di sebelah kiri dipetakan ke jadwal pulsa gelombang mikro di kanan atas. Kanan bawah menggambarkan jadwal yang sama, namun dengan rangkaian dua pulsa X yang disisipkan selama periode diam Qubit pertama.

![Gambaran dynamical decoupling](../docs/images/guides/error-mitigation-explanation/dd.avif)

Dynamical decoupling dapat diaktifkan dengan mengatur `enable` ke `True` pada [opsi dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Opsi `sequence_type` dapat digunakan untuk memilih dari beberapa rangkaian pulsa yang berbeda. Tipe rangkaian default adalah `"XX"`.

Sel kode berikut menunjukkan cara mengaktifkan dynamical decoupling untuk Estimator dan memilih rangkaian dynamical decoupling.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Pauli twirling
Twirling, juga dikenal sebagai [randomized compiling](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), adalah teknik yang banyak digunakan untuk mengubah kanal noise arbitrer menjadi kanal noise dengan struktur yang lebih spesifik.

Pauli twirling adalah jenis twirling khusus yang menggunakan operasi Pauli. Teknik ini berfungsi untuk mengubah kanal kuantum apa pun menjadi kanal Pauli. Dilakukan sendiri, teknik ini dapat memitigasi noise koheren karena noise koheren cenderung terakumulasi secara kuadratik seiring jumlah operasi, sedangkan noise Pauli terakumulasi secara linier. Pauli twirling sering dikombinasikan dengan teknik mitigasi error lain yang bekerja lebih baik dengan noise Pauli daripada noise arbitrer.

Pauli twirling diimplementasikan dengan men挟pit sekumpulan Gate yang dipilih dengan Gate Pauli satu Qubit yang dipilih secara acak sedemikian rupa sehingga efek ideal dari Gate tetap sama. Hasilnya adalah satu Circuit digantikan dengan ansambel Circuit acak, semuanya dengan efek ideal yang sama. Saat mengambil sampel Circuit, sampel diambil dari beberapa instance acak, bukan hanya satu.

![Gambaran Pauli twirling](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Karena sebagian besar error pada perangkat keras kuantum saat ini berasal dari Gate dua Qubit, teknik ini sering diterapkan secara eksklusif pada Gate dua Qubit (asli). Diagram berikut menggambarkan beberapa Pauli twirl untuk Gate CNOT dan ECR. Setiap Circuit dalam satu baris memiliki efek ideal yang sama.

![Gambaran twirl Gate](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Pauli twirling dapat diaktifkan dengan mengatur `enable_gates` ke `True` pada [opsi twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Opsi penting lainnya meliputi:

- `num_randomizations`: Jumlah instance Circuit yang diambil dari ansambel Circuit yang di-twirl.
- `shots_per_randomization`: Jumlah shot yang diambil dari setiap instance Circuit.

Sel kode berikut menunjukkan cara mengaktifkan Pauli twirling dan mengatur opsi-opsi ini untuk estimator. Tidak ada opsi yang perlu diatur secara eksplisit.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Twirled readout error extinction (TREX)
[Twirled readout error extinction (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) memitigasi efek error pengukuran untuk estimasi nilai ekspektasi observabel Pauli.
Teknik ini didasarkan pada konsep twirled measurements, yang dilakukan dengan secara acak mengganti Gate pengukuran dengan rangkaian (1) Gate Pauli X, (2) pengukuran, dan (3) bit flip klasik. Sama seperti gate twirling standar, rangkaian ini setara dengan pengukuran biasa tanpa adanya noise, seperti yang digambarkan dalam diagram berikut:

![Gambaran measurement twirling](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

Dengan adanya readout error, measurement twirling berfungsi untuk mendiagonalisasi matriks transfer readout-error, sehingga mudah untuk diinversikan. Mengestimasi matriks transfer readout-error memerlukan eksekusi Circuit kalibrasi tambahan, yang menimbulkan sedikit overhead.

TREX dapat diaktifkan dengan mengatur `measure_mitigation` ke `True` pada [opsi resiliensi Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) untuk Estimator. Opsi untuk pembelajaran noise pengukuran dijelaskan [di sini](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Seperti gate twirling, kamu dapat mengatur jumlah randomisasi Circuit dan jumlah shot per randomisasi.

Sel kode berikut menunjukkan cara mengaktifkan TREX dan mengatur opsi-opsi ini untuk estimator. Tidak ada opsi yang perlu diatur secara eksplisit.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Zero-noise extrapolation (ZNE)
Zero-noise extrapolation (ZNE) adalah teknik untuk memitigasi error dalam mengestimasi nilai ekspektasi observabel. Meskipun sering meningkatkan hasil, teknik ini tidak dijamin menghasilkan hasil yang tidak bias.

ZNE terdiri dari dua tahap:

1. _Amplifikasi noise_: Circuit kuantum asli dieksekusi beberapa kali pada tingkat noise yang berbeda.
2. _Ekstrapolasi_: Hasil ideal diestimasi dengan mengekstrapolasi hasil nilai ekspektasi yang noisy ke batas zero-noise.

Baik tahap amplifikasi noise maupun ekstrapolasi dapat diimplementasikan dengan berbagai cara. Qiskit Runtime mengimplementasikan amplifikasi noise dengan "digital gate folding," yang berarti Gate dua Qubit digantikan dengan rangkaian ekuivalen dari Gate dan inversnya. Misalnya, menggantikan uniter $U$ dengan $U U^\dagger U$ akan menghasilkan faktor amplifikasi noise sebesar 3. Untuk ekstrapolasi, kamu dapat memilih dari salah satu bentuk fungsional, termasuk fit linier atau fit eksponensial.
Gambar di bawah ini menggambarkan digital gate folding di sebelah kiri, dan prosedur ekstrapolasi di sebelah kanan.

![Gambaran ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE dapat diaktifkan dengan mengatur `zne_mitigation` ke `True` pada [opsi resiliensi Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) untuk Estimator.
Opsi Qiskit Runtime untuk ZNE dijelaskan [di sini](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Opsi-opsi penting berikut ini:

- `noise_factors`: Faktor noise yang digunakan untuk amplifikasi noise.
- `extrapolator`: Bentuk fungsional yang digunakan untuk ekstrapolasi.

Sel kode berikut menunjukkan cara mengaktifkan ZNE dan mengatur opsi-opsi ini untuk estimator. Tidak ada opsi yang perlu diatur secara eksplisit.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Probabilistic Error Amplification (PEA)
Salah satu tantangan utama dalam ZNE adalah mengamplifikasi noise yang mempengaruhi Circuit target secara akurat. Gate folding menyediakan cara mudah untuk melakukan amplifikasi ini, namun berpotensi tidak akurat dan dapat menghasilkan hasil yang salah. Lihat artikel ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), khususnya halaman 4 dari informasi tambahan untuk detailnya. Probabilistic error amplification memberikan pendekatan yang lebih akurat untuk amplifikasi error melalui pembelajaran noise.

PEA adalah teknik yang lebih canggih yang melakukan eksperimen pendahuluan untuk merekonstruksi noise dan kemudian menggunakan informasi ini untuk melakukan amplifikasi yang akurat. Teknik ini dimulai dengan mempelajari model noise yang di-twirl dari setiap lapisan Gate entangling dalam Circuit sebelum dijalankan (lihat [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) untuk opsi pembelajaran yang relevan). Setelah fase pembelajaran, Circuit dieksekusi pada setiap faktor noise, di mana setiap lapisan entangling dari Circuit diamplifikasi dengan menyuntikkan secara probabilistik noise satu Qubit yang proporsional dengan model noise yang dipelajari. Lihat artikel ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) untuk detail lebih lanjut.

PEA terdiri dari tiga tahap:
1. _Pembelajaran_: Model noise yang di-twirl dari setiap lapisan Gate entangling dalam Circuit dipelajari.
1. _Amplifikasi noise_: Circuit kuantum asli dieksekusi beberapa kali pada faktor noise yang berbeda.
2. _Ekstrapolasi_: Hasil ideal diestimasi dengan mengekstrapolasi hasil nilai ekspektasi yang noisy ke batas zero-noise.

Untuk eksperimen skala utilitas, PEA sering menjadi pilihan terbaik.

Karena PEA adalah teknik amplifikasi noise ZNE, kamu juga perlu mengaktifkan ZNE dengan mengatur `resilience.zne_mitigation = True`. Opsi [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) lainnya juga dapat digunakan untuk mengatur ekstrapolator, level amplifikasi, dan sebagainya. PEA memerlukan model noise, yang secara otomatis dihasilkan saat menggunakan primitif.

Cuplikan berikut memberikan contoh di mana PEA digunakan untuk memitigasi hasil job Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Probabilistic error cancellation (PEC)
Probabilistic error cancellation (PEC) adalah teknik untuk memitigasi error dalam mengestimasi nilai ekspektasi observabel. Tidak seperti ZNE, teknik ini mengembalikan estimasi nilai ekspektasi yang tidak bias. Namun, umumnya menimbulkan overhead yang lebih besar.

Dalam PEC, efek dari Circuit target ideal dinyatakan sebagai kombinasi linier dari Circuit berisik yang sebenarnya dapat diimplementasikan dalam praktik:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Output dari Circuit ideal kemudian dapat direproduksi dengan mengeksekusi berbagai instance Circuit berisik yang diambil dari ansambel acak yang didefinisikan oleh kombinasi linier. Jika koefisien $\eta_i$ membentuk distribusi probabilitas, koefisien tersebut dapat digunakan langsung sebagai probabilitas ansambel. Dalam praktiknya, beberapa koefisien bernilai negatif, sehingga membentuk distribusi kuasi-probabilitas. Koefisien tersebut masih dapat digunakan untuk mendefinisikan ansambel acak, namun ada overhead pengambilan sampel yang terkait dengan negativitas distribusi kuasi-probabilitas, yang dikarakterisasi oleh besaran

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Overhead pengambilan sampel adalah faktor perkalian pada jumlah shot yang diperlukan untuk mengestimasi nilai ekspektasi hingga presisi tertentu, dibandingkan dengan jumlah shot yang dibutuhkan dari Circuit ideal. Hal ini berskala secara kuadratik dengan $\gamma$, yang pada gilirannya berskala secara eksponensial dengan kedalaman Circuit.

PEC dapat diaktifkan dengan mengatur `pec_mitigation` ke `True` pada [opsi resiliensi Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) untuk Estimator.
Opsi Qiskit Runtime untuk PEC dijelaskan [di sini](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Batas pada overhead pengambilan sampel dapat diatur menggunakan opsi `max_overhead`. Perlu diperhatikan bahwa membatasi overhead pengambilan sampel dapat menyebabkan presisi hasil melebihi presisi yang diminta. Nilai default `max_overhead` adalah 100.

Sel kode berikut menunjukkan cara mengaktifkan PEC dan mengatur opsi `max_overhead` untuk estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Langkah selanjutnya
- Lihat [tutorial](/tutorials/combine-error-mitigation-techniques) tentang menggabungkan opsi mitigasi error dengan primitif Estimator.
- [Konfigurasi mitigasi error.](configure-error-mitigation)
- [Konfigurasi supresi error.](configure-error-suppression)
- Jelajahi [opsi](runtime-options-overview) lain untuk primitif Qiskit Runtime.
- Tentukan [mode eksekusi](execution-modes) mana yang akan digunakan untuk menjalankan job kamu.